# Orbital Debris Database: Building the Analytical SQLite Source
 
**Datasets:**
- kinetic_master.csv (cleaned, merged, and physics-enriched object list)
 
**Objective:** Create a normalized SQLite database from the kinetic master file, with tables designed for efficient queries and visualizations.
 
### Why this notebook?
The project needs a single, query-ready database that brings together all cleaned and derived orbital object data. This notebook takes the master CSV and builds a normalized SQLite database for analysis and visualization.
 
### What we do here
1. **Load the master dataset:** Read in the cleaned kinetic_master.csv file.
2. **Design the schema:** Decide on tables, primary keys, and relationships for efficient queries.
3. **Clean and patch metadata:** Standardize and fill in missing values, especially for ownership and launch details.
4. **Build and export tables:** Create normalized tables and write them to SQLite.
5. **Run validation checks:** Confirm data integrity and schema alignment.
 
This sets up the foundation for all downstream queries, charts, and risk modeling.

In [ ]:
import pandas as pd
import sqlite3
import utility as utils

df_master = pd.read_csv('../data/clean/kinetic_master.csv', low_memory=False)

df = df_master.copy()

conn = sqlite3.connect('../data/clean/orbital_debris.db')

### Stage 1.0: Standardize Ownership Fields
 
**The issue:**
- Owner codes and names in the master dataset have inconsistent formatting and naming conventions, especially for major operators like SpaceX.
- Inconsistent owner fields can cause join errors and reduce data quality.
 
**What we do:**
- Strip whitespace and standardize case for `owner_code` and `owner`.
- Map common variations of SpaceX and related names to a single canonical form.
 
**Why it matters:**
- Ensures all ownership fields are consistent and ready for reliable joins and grouping in downstream tables.

In [ ]:
# Standardize owner_code and owner fields
df['owner_code'] = df['owner_code'].astype(str).str.strip().str.upper()
df['owner'] = df['owner'].astype(str).str.strip()

owner_name_map = {
    'Spacex': 'SpaceX',
    'spacex': 'SpaceX',
    'spaceX': 'SpaceX',
    'Swarm Technologies': 'SpaceX',
    'Space Exploration Technologies Corp.': 'SpaceX'
}

df['owner'] = df['owner'].replace(owner_name_map)

### Stage 1.1: Aggregate Ownership Metadata
 
**The issue:**
- Boolean sector flags (commercial, government, military, civil) may have mixed types or missing values, making analysis unreliable.
- Ownership metadata is spread across multiple rows and needs to be aggregated for normalization.
 
**What we do:**
- Coerce all flag columns to consistent 0/1 numeric types.
- Aggregate ownership metadata by `owner_code`, taking the first non-null value for string fields and the max for boolean flags.
- Create a unique, joinable `ownership_operators` table for the database.
 
**Why it matters:**
- Ensures sector flags are reliable for analysis and visualization.
- Aggregated ownership metadata enables efficient joins and reduces redundancy in the database.

In [ ]:
# Coerce flag columns to 0/1 and aggregate ownership metadata
flag_cols = ['is_commercial', 'is_government', 'is_military', 'is_civil']

for col in flag_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

owner_profile = df.groupby('owner_code', as_index=False).agg({
    'owner': utils.first_non_null,
    'country_operator': utils.first_non_null,
    'users': utils.first_non_null,
    'is_commercial': 'max',
    'is_government': 'max',
    'is_military': 'max',
    'is_civil': 'max',
    'contractor': utils.first_non_null,
    'contractor_country': utils.first_non_null
})


### Stage 1.2: Fix Owner Profile Flags and Derive Per-Object User Category

**The issue:**
- The `max` aggregation used for boolean sector flags in Stage 1.1 pollutes owner entries. Any `owner_code` with even one commercial object (e.g., Starlink and NOAA both share `owner_code='US'`) inherits `is_commercial=1` for the entire group, regardless of the majority use.
- The `first_non_null` aggregation for `users` text is order-dependent and may pick an unrepresentative minority value.

**What we do:**
- Re-compute `owner_profile.users` using the **mode** (most common non-null value) per `owner_code` — more representative than the first encountered value.
- Re-derive all four sector flags from the mode-based `users` text — ensuring flags are consistent with the text field.
- Compute a `user_category` column **per object** in `df`, using per-object UCS flags and text as the primary source, with the owner-level mode-based category as a fallback for SATCAT-only objects that lack UCS data.

**Why it matters:**
- Per-object `user_category` is far more accurate than an owner-level lookup for mixed-sector owner codes.
- Storing it directly in the `satellites` table eliminates the need for a brittle join-based CASE expression in analysis queries.


In [ ]:
# Re-compute users text using mode per owner_code (more representative than first_non_null).
# first_non_null is order-dependent and can pick a minority value (e.g., 'Civil' for 'US'
# even though 'Commercial' is the majority).
users_mode = (
    df.groupby('owner_code')['users']
    .agg(lambda x: x.dropna().mode().iloc[0] if x.dropna().shape[0] > 0 else None)
    .rename('users')
)

owner_profile = owner_profile.set_index('owner_code')
owner_profile['users'] = users_mode
owner_profile = owner_profile.reset_index()

# Re-derive boolean flags from the mode-based users text.
# This eliminates the max() aggregation artifact where is_commercial=1 bleeds across
# mixed-sector owner codes like 'US'.
owner_profile['is_commercial'] = owner_profile['users'].str.contains('Commercial', case=False, na=False).astype(int)
owner_profile['is_government'] = owner_profile['users'].str.contains('Government', case=False, na=False).astype(int)
owner_profile['is_military']   = owner_profile['users'].str.contains('Military',   case=False, na=False).astype(int)
owner_profile['is_civil']      = owner_profile['users'].str.contains('Civil',      case=False, na=False).astype(int)

owner_category_map = (
    owner_profile
    .set_index('owner_code')['users']
    .map(utils.category_from_users_text)
    .to_dict()
)

# Derive user_category PER OBJECT using a three-tier priority:
df['user_category'] = df.apply(utils.derive_user_category, axis=1, map=owner_category_map)

# Quick validation
in_orbit_cat = df[df['in_orbit'] == 1]['user_category'].value_counts()
print("Per-object user_category for in-orbit objects:")
print(in_orbit_cat.to_string())
print(f"\nTotal in-orbit: {in_orbit_cat.sum():,}")


### Stage 1.3: Patch and Fill Key Metadata
 
**The issue:**
- Some fields (e.g., `primary_purpose`, `un_registry`, `lifetime_years`, `orbit_type`, `launch_id`) are missing or inconsistent, especially for non-payload objects.
- Incomplete or inconsistent metadata can cause errors in downstream analysis and reduce data quality.
 
**What we do:**
- For non-payload objects, fill missing `primary_purpose` and `un_registry` with 'Not Applicable'.
- Ensure `lifetime_years` is numeric and nullable (no forced imputation).
- Fill missing `orbit_type` with 'Other/Misc'.
- Synthesize `launch_id` from the COSPAR prefix (YYYY-NNN), filling missing values with 'UNKNOWN'.
 
**Why it matters:**
- Ensures all key fields are complete and consistent, supporting robust queries and analysis in the final database.

In [ ]:
is_payload = df['object_type'].astype(str).str.strip().str.upper().eq('PAYLOAD')

df.loc[~is_payload, 'primary_purpose'] = df.loc[~is_payload, 'primary_purpose'].fillna('Not Applicable')
df.loc[~is_payload, 'un_registry'] = df.loc[~is_payload, 'un_registry'].fillna('Not Applicable')

df['lifetime_years'] = pd.to_numeric(df['lifetime_years'], errors='coerce')
df['orbit_type'] = df['orbit_type'].fillna('Other/Misc')
df['launch_id'] = df['cospar_id'].astype(str).str.extract(r'^(\d{4}-\d{3})', expand=False)
df['launch_id'] = df['launch_id'].fillna('UNKNOWN')

### Stage 2.0: Build and Export Database Tables

**The issue:**
- The master DataFrame contains all orbital object data, but analysis and visualization require normalized, query-ready tables in SQLite.
- Without normalization, queries are slow, error-prone, and difficult to maintain.

**What we do:**
- Build individual DataFrames for each logical table (satellites, orbital data, ownership, launches, etc.).
- Export each DataFrame to SQLite with schema-aligned table names.
- Run validation checks to ensure data integrity and schema alignment.

**Why it matters:**
- Normalized tables enable efficient queries, reduce redundancy, and support robust analysis and visualization.
- Validation ensures the exported database is reliable for all downstream work.

In [ ]:

# build the individual tables for SQLite export, selecting relevant columns and dropping duplicates where necessary.
df_ownership_operators = owner_profile[
    ['owner_code', 'owner', 'country_operator', 'users',
     'is_commercial', 'is_government', 'is_military', 'is_civil',
     'contractor', 'contractor_country']
 ]

df_launch_events = df[
    ['launch_id', 'launch_date', 'launch_year', 'launch_site']
].drop_duplicates(subset=['launch_id'])

# user_category is now a per-object column derived in Stage 1.2
df_satellites = df[
    ['norad_id', 'cospar_id', 'object_name', 'satellite_name', 'official_name',
     'object_type', 'category', 'ops_status', 'data_status', 'decay_date', 'in_orbit',
     'owner_code', 'launch_id', 'user_category']
].drop_duplicates(subset=['norad_id'])

df_orbital_data = df[
    ['norad_id', 'orbit_class', 'orbit_type', 'period_minutes', 'perigee_km',
     'apogee_km', 'inclination_degrees', 'eccentricity', 'semi_major_axis_km',
     'launch_mass_kg', 'proxy_mass_kg', 'dry_mass_kg', 'power_watts',
     'proxy_power_watts', 'rcs', 'rcs_class']
].drop_duplicates(subset=['norad_id'])

df_ucs_details = df[
    ['norad_id', 'lifetime_years', 'sat_age_years',
     'primary_purpose', 'detailed_purpose', 'geo_longitude', 'un_registry']
].drop_duplicates(subset=['norad_id'])

df_risk_assessment = df[
    ['norad_id', 'velocity_kms', 'kinetic_joules', 'is_zombie']
].drop_duplicates(subset=['norad_id'])

# write each dataframe to SQLite using schema-aligned table names
df_satellites.to_sql('satellites', conn, if_exists='replace', index=False)
df_orbital_data.to_sql('orbital_data', conn, if_exists='replace', index=False)
df_ucs_details.to_sql('ucs_details', conn, if_exists='replace', index=False)
df_risk_assessment.to_sql('risk_assessment', conn, if_exists='replace', index=False)
df_ownership_operators.to_sql('ownership_operators', conn, if_exists='replace', index=False)
df_launch_events.to_sql('launch_events', conn, if_exists='replace', index=False)


### Stage 3: Sanity Checks and Validation

**The issue:**
- Exported tables may have missing keys, duplicates, or schema mismatches that can break downstream queries.

**What we do:**
- For each table, load from SQLite and run `utils.quick_report` to check for nulls, duplicates, and schema alignment.
- Commit and close the database connection after validation.

**Why it matters:**
- Ensures the exported database is reliable, complete, and ready for analysis and visualization.

In [ ]:
primary_keys = {
    'satellites': 'norad_id',
    'orbital_data': 'norad_id',
    'ucs_details': 'norad_id',
    'risk_assessment': 'norad_id',
    'ownership_operators': 'owner_code',
    'launch_events': 'launch_id'
}

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)['name']

for table in tables:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    key_col = primary_keys.get(table)
    utils.quick_report(df, title=f"Table: {table}", key_col=key_col)

conn.commit()
conn.close()

print('\nSQLite build complete: ../data/clean/orbital_debris.db')